# A03 — Multiclass ROC Analysis

**EEEM068 Applied Machine Learning — Supplementary ROC/AUC Evaluation**

**Purpose:** Produce report/appendix-ready multiclass ROC curves for the three required core models:

1. EfficientNet-B4 — baseline
2. DeiT-III-B/16 — previous main-assessment model
3. MaxViT-Tiny M12 — new LSA model

This notebook is **analysis only**. It does **not** train, fine-tune, select, or modify any model.

It loads the frozen test prediction CSV files already produced by A01 and evaluates all three models on the **same frozen internal test image set**.

### ROC protocol

Because diabetic retinopathy grading is a five-class problem, ROC is calculated using a **one-vs-rest (OvR)** formulation:

- one ROC curve and AUC for each DR grade;
- micro-average ROC/AUC;
- macro-average ROC/AUC;
- a three-panel comparison figure for the appendix.

ROC/AUC is supplementary. The primary ordinal metric remains **quadratic weighted kappa (QWK)**.

## 1. Imports and paths

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    auc,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize

print("Imports OK.")

In [ ]:
NUM_CLASSES = 5

CLASS_NAMES = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR",
]

PROJECT_ROOT = Path(
    "/scratch/New AML/EEEM068-LSA-Diabetic-Retinopathy"
)

A03_LOG_DIR = (
    PROJECT_ROOT
    / "logs"
    / "analysis"
    / "A03_multiclass_roc_analysis"
)

A03_FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "analysis"
    / "A03_multiclass_roc_analysis"
)

for directory in [A03_LOG_DIR, A03_FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MODEL_PREDICTION_FILES = {
    "EfficientNet-B4": (
        PROJECT_ROOT
        / "logs"
        / "efficientnet_b4"
        / "exp01_shared_p0_wrs_focal_seed42"
        / "A01_frozen_test_predictions"
        / "test_predictions.csv"
    ),
    "DeiT-III": (
        PROJECT_ROOT
        / "logs"
        / "deit3_b16"
        / "D07_two_phase_finetuning"
        / "A01_frozen_test_predictions"
        / "test_predictions.csv"
    ),
    "MaxViT-Tiny M12": (
        PROJECT_ROOT
        / "logs"
        / "maxvit_tiny"
        / "M12_combined_fine_grained_guidance"
        / "A01_frozen_test_predictions"
        / "test_predictions.csv"
    ),
}

print(f"A03 logs    -> {A03_LOG_DIR}")
print(f"A03 figures -> {A03_FIGURE_DIR}")

for model_name, path in MODEL_PREDICTION_FILES.items():
    print(f"{model_name:20s} -> {path}")

## 2. Load the frozen A01 test predictions

In [ ]:
PROBABILITY_COLUMNS = [
    f"prob_grade_{class_id}"
    for class_id in range(NUM_CLASSES)
]

REQUIRED_COLUMNS = {
    "image_id",
    "true_label",
    *PROBABILITY_COLUMNS,
}

prediction_tables = {}

for model_name, prediction_path in MODEL_PREDICTION_FILES.items():
    if not prediction_path.exists():
        raise FileNotFoundError(
            f"{model_name} frozen A01 prediction file was not found:\n"
            f"{prediction_path}\n\n"
            "A03 does not retrain models. The A01 frozen prediction outputs "
            "must exist before ROC analysis can run."
        )

    table = pd.read_csv(prediction_path)
    table["image_id"] = table["image_id"].astype(str)

    missing_columns = REQUIRED_COLUMNS.difference(table.columns)

    if missing_columns:
        raise KeyError(
            f"{model_name} prediction file is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    if table["image_id"].duplicated().any():
        raise ValueError(
            f"{model_name} contains duplicate image IDs."
        )

    probabilities = table[PROBABILITY_COLUMNS].to_numpy(dtype=float)

    if not np.isfinite(probabilities).all():
        raise ValueError(
            f"{model_name} contains non-finite probabilities."
        )

    if not np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-5,
    ):
        raise ValueError(
            f"{model_name} probabilities do not sum to approximately 1."
        )

    prediction_tables[model_name] = table

    print(
        f"{model_name}: loaded {len(table):,} frozen test predictions."
    )

## 3. Verify identical held-out images and labels

In [ ]:
CORE_MODELS = [
    "EfficientNet-B4",
    "DeiT-III",
    "MaxViT-Tiny M12",
]

reference_model = CORE_MODELS[0]
reference_table = prediction_tables[reference_model].copy()

reference_ids = reference_table["image_id"].tolist()
reference_id_set = set(reference_ids)

for model_name in CORE_MODELS[1:]:
    table = prediction_tables[model_name]

    if set(table["image_id"]) != reference_id_set:
        raise AssertionError(
            f"{model_name} does not contain exactly the same test image IDs "
            f"as {reference_model}."
        )

# Align every model to the EfficientNet reference order.
aligned_tables = {}

for model_name in CORE_MODELS:
    table = (
        prediction_tables[model_name]
        .set_index("image_id")
        .loc[reference_ids]
        .reset_index()
    )

    aligned_tables[model_name] = table

reference_labels = aligned_tables[reference_model][
    "true_label"
].to_numpy(dtype=int)

for model_name in CORE_MODELS[1:]:
    model_labels = aligned_tables[model_name][
        "true_label"
    ].to_numpy(dtype=int)

    if not np.array_equal(
        model_labels,
        reference_labels,
    ):
        raise AssertionError(
            f"{model_name} labels differ from {reference_model} "
            "after image-ID alignment."
        )

print("Identical test image IDs : PASSED")
print("Identical test labels    : PASSED")
print(f"Common test images       : {len(reference_ids):,}")
print(
    "Class counts             :",
    np.bincount(reference_labels, minlength=NUM_CLASSES).tolist(),
)

## 4. Compute one-vs-rest ROC curves and AUC

For each model, each DR grade is treated as the positive class in turn while the remaining four grades form the negative class.

The notebook also reports:

- **micro-average AUC**, which pools all one-vs-rest decisions;
- **macro-average AUC**, which gives each class equal weight.

Macro-AUC is useful in this imbalanced dataset because it does not allow the majority class to dominate the summary as strongly as a prevalence-weighted measure.

In [ ]:
def compute_multiclass_roc(
    labels,
    probabilities,
    class_names,
):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)

    y_binary = label_binarize(
        labels,
        classes=np.arange(len(class_names)),
    )

    fpr = {}
    tpr = {}
    thresholds = {}
    per_class_auc = {}

    for class_id, class_name in enumerate(class_names):
        class_fpr, class_tpr, class_thresholds = roc_curve(
            y_binary[:, class_id],
            probabilities[:, class_id],
        )

        fpr[class_id] = class_fpr
        tpr[class_id] = class_tpr
        thresholds[class_id] = class_thresholds
        per_class_auc[class_id] = auc(
            class_fpr,
            class_tpr,
        )

    micro_fpr, micro_tpr, micro_thresholds = roc_curve(
        y_binary.ravel(),
        probabilities.ravel(),
    )

    micro_auc = auc(
        micro_fpr,
        micro_tpr,
    )

    # Macro-average ROC curve:
    # interpolate every class TPR over the union of class FPR positions.
    all_fpr = np.unique(
        np.concatenate(
            [fpr[class_id] for class_id in range(len(class_names))]
        )
    )

    mean_tpr = np.zeros_like(all_fpr)

    for class_id in range(len(class_names)):
        mean_tpr += np.interp(
            all_fpr,
            fpr[class_id],
            tpr[class_id],
        )

    mean_tpr /= len(class_names)

    macro_fpr = all_fpr
    macro_tpr = mean_tpr
    macro_auc_curve = auc(
        macro_fpr,
        macro_tpr,
    )

    # Independent sklearn macro-OvR AUC as a consistency check.
    macro_auc_sklearn = roc_auc_score(
        y_binary,
        probabilities,
        average="macro",
        multi_class="ovr",
    )

    return {
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "per_class_auc": per_class_auc,
        "micro_fpr": micro_fpr,
        "micro_tpr": micro_tpr,
        "micro_thresholds": micro_thresholds,
        "micro_auc": micro_auc,
        "macro_fpr": macro_fpr,
        "macro_tpr": macro_tpr,
        "macro_auc": macro_auc_curve,
        "macro_auc_sklearn": macro_auc_sklearn,
    }


roc_results = {}

for model_name in CORE_MODELS:
    table = aligned_tables[model_name]

    probabilities = table[
        PROBABILITY_COLUMNS
    ].to_numpy(dtype=float)

    roc_results[model_name] = compute_multiclass_roc(
        reference_labels,
        probabilities,
        CLASS_NAMES,
    )

    result = roc_results[model_name]

    print(
        f"{model_name}: "
        f"macro AUC={result['macro_auc']:.4f}, "
        f"micro AUC={result['micro_auc']:.4f}"
    )

    if not np.isclose(
        result["macro_auc"],
        result["macro_auc_sklearn"],
        atol=5e-3,
    ):
        print(
            "  Note: interpolated macro-ROC AUC and sklearn "
            "macro-OvR AUC differ slightly because they are "
            "computed by related but not identical averaging procedures."
        )

## 5. Save AUC summary tables

In [ ]:
summary_rows = []
per_class_rows = []

for model_name in CORE_MODELS:
    result = roc_results[model_name]

    summary_rows.append(
        {
            "model": model_name,
            "macro_auc": result["macro_auc"],
            "macro_auc_sklearn_ovr": result[
                "macro_auc_sklearn"
            ],
            "micro_auc": result["micro_auc"],
            "n_test_images": len(reference_labels),
        }
    )

    for class_id, class_name in enumerate(CLASS_NAMES):
        per_class_rows.append(
            {
                "model": model_name,
                "class_id": class_id,
                "class_name": class_name,
                "auc_ovr": result["per_class_auc"][
                    class_id
                ],
                "support": int(
                    np.sum(reference_labels == class_id)
                ),
            }
        )

auc_summary_df = pd.DataFrame(summary_rows)
per_class_auc_df = pd.DataFrame(per_class_rows)

display(
    auc_summary_df.style.format(
        {
            "macro_auc": "{:.4f}",
            "macro_auc_sklearn_ovr": "{:.4f}",
            "micro_auc": "{:.4f}",
        }
    )
)

display(
    per_class_auc_df.style.format(
        {
            "auc_ovr": "{:.4f}",
        }
    )
)

auc_summary_path = (
    A03_LOG_DIR / "multiclass_auc_summary.csv"
)

per_class_auc_path = (
    A03_LOG_DIR / "per_class_auc.csv"
)

auc_summary_df.to_csv(
    auc_summary_path,
    index=False,
)

per_class_auc_df.to_csv(
    per_class_auc_path,
    index=False,
)

print(f"Saved -> {auc_summary_path}")
print(f"Saved -> {per_class_auc_path}")

## 6. Individual multiclass ROC figures

In [ ]:
def plot_model_roc(
    model_name,
    result,
    output_stem,
):
    fig, ax = plt.subplots(
        figsize=(7.4, 6.2)
    )

    for class_id, class_name in enumerate(CLASS_NAMES):
        ax.plot(
            result["fpr"][class_id],
            result["tpr"][class_id],
            linewidth=1.8,
            label=(
                f"{class_name} "
                f"(AUC={result['per_class_auc'][class_id]:.3f})"
            ),
        )

    ax.plot(
        result["macro_fpr"],
        result["macro_tpr"],
        linewidth=2.6,
        linestyle="--",
        label=(
            f"Macro-average "
            f"(AUC={result['macro_auc']:.3f})"
        ),
    )

    ax.plot(
        result["micro_fpr"],
        result["micro_tpr"],
        linewidth=2.2,
        linestyle=":",
        label=(
            f"Micro-average "
            f"(AUC={result['micro_auc']:.3f})"
        ),
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.0,
        label="Chance",
    )

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.02)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(
        f"{model_name}: one-vs-rest ROC curves",
        fontweight="bold",
    )
    ax.grid(alpha=0.25)
    ax.legend(
        loc="lower right",
        fontsize=8,
    )

    fig.tight_layout()

    png_path = (
        A03_FIGURE_DIR / f"{output_stem}.png"
    )
    pdf_path = (
        A03_FIGURE_DIR / f"{output_stem}.pdf"
    )

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    plt.show()

    print(f"Saved -> {png_path}")
    print(f"Saved -> {pdf_path}")


plot_model_roc(
    "EfficientNet-B4",
    roc_results["EfficientNet-B4"],
    "A03_figure_01_efficientnet_roc",
)

plot_model_roc(
    "DeiT-III-B/16",
    roc_results["DeiT-III"],
    "A03_figure_02_deit_roc",
)

plot_model_roc(
    "MaxViT-Tiny M12",
    roc_results["MaxViT-Tiny M12"],
    "A03_figure_03_maxvit_m12_roc",
)

## 7. Report/appendix-ready comparison figure

The following three-panel figure is the recommended ROC visualisation for the appendix because it compares the three required core models on the same held-out data while avoiding an overcrowded single axis.

In [ ]:
display_names = {
    "EfficientNet-B4": "EfficientNet-B4",
    "DeiT-III": "DeiT-III-B/16",
    "MaxViT-Tiny M12": "MaxViT-Tiny M12",
}

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5.4),
    sharex=True,
    sharey=True,
)

for panel_index, model_name in enumerate(CORE_MODELS):
    ax = axes[panel_index]
    result = roc_results[model_name]

    for class_id, class_name in enumerate(CLASS_NAMES):
        ax.plot(
            result["fpr"][class_id],
            result["tpr"][class_id],
            linewidth=1.5,
            label=(
                f"{class_name} "
                f"({result['per_class_auc'][class_id]:.3f})"
            ),
        )

    ax.plot(
        result["macro_fpr"],
        result["macro_tpr"],
        linewidth=2.5,
        linestyle="--",
        label=(
            f"Macro "
            f"({result['macro_auc']:.3f})"
        ),
    )

    ax.plot(
        result["micro_fpr"],
        result["micro_tpr"],
        linewidth=2.0,
        linestyle=":",
        label=(
            f"Micro "
            f"({result['micro_auc']:.3f})"
        ),
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=0.9,
        label="Chance",
    )

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.02)
    ax.set_xlabel("False Positive Rate")
    ax.set_title(
        f"({chr(97 + panel_index)}) "
        f"{display_names[model_name]}",
        fontweight="bold",
    )
    ax.grid(alpha=0.22)

    if panel_index == 0:
        ax.set_ylabel("True Positive Rate")

    ax.legend(
        loc="lower right",
        fontsize=7,
    )

fig.suptitle(
    "Multiclass one-vs-rest ROC comparison on the frozen internal test set",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)

fig.tight_layout()

comparison_png = (
    A03_FIGURE_DIR
    / "A03_figure_04_core_model_roc_comparison.png"
)

comparison_pdf = (
    A03_FIGURE_DIR
    / "A03_figure_04_core_model_roc_comparison.pdf"
)

fig.savefig(
    comparison_png,
    dpi=350,
    bbox_inches="tight",
)

fig.savefig(
    comparison_pdf,
    bbox_inches="tight",
)

plt.show()

print(f"Saved -> {comparison_png}")
print(f"Saved -> {comparison_pdf}")

## 8. Compact interpretation table for report writing

In [ ]:
best_macro_model = (
    auc_summary_df
    .sort_values(
        "macro_auc",
        ascending=False,
    )
    .iloc[0]
)

best_micro_model = (
    auc_summary_df
    .sort_values(
        "micro_auc",
        ascending=False,
    )
    .iloc[0]
)

print(
    "Highest macro-AUC:",
    best_macro_model["model"],
    f"({best_macro_model['macro_auc']:.4f})",
)

print(
    "Highest micro-AUC:",
    best_micro_model["model"],
    f"({best_micro_model['micro_auc']:.4f})",
)

print(
    "\nImportant interpretation note:\n"
    "ROC/AUC measures discrimination under one-vs-rest thresholding. "
    "It does not encode the ordinal distance between DR grades. "
    "Therefore QWK remains the primary model-selection metric, "
    "with macro-F1, balanced accuracy, class-wise performance, "
    "calibration, failure analysis and explainability used alongside it."
)

## 9. Suggested appendix caption

**Figure A?.** Multiclass one-vs-rest receiver operating characteristic (ROC) curves for EfficientNet-B4, DeiT-III-B/16 and MaxViT-Tiny M12 on the common frozen internal test set. Curves are shown for each diabetic-retinopathy grade together with macro- and micro-average ROC curves. AUC is treated as supplementary discrimination evidence because it does not account for the ordinal distance between grading errors; QWK remains the primary model-selection metric.

## 10. Completion checklist

After running this notebook once:

- confirm all three prediction files were loaded rather than any model being retrained;
- confirm identical image IDs and labels passed;
- record the macro/micro/per-class AUC values;
- use `A03_figure_04_core_model_roc_comparison.pdf` or `.png` in the appendix;
- commit this executed notebook together with `logs/analysis/A03_multiclass_roc_analysis/` and `results/figures/analysis/A03_multiclass_roc_analysis/`.